In [6]:
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, WeightedRandomSampler
import wandb
from sklearn.metrics import precision_score
from accelerate import Accelerator
from accelerate import DistributedType, DistributedDataParallelKwargs
from peft import get_peft_model, LoraConfig, TaskType
import os
from utils.utils import seed_everything
from transformers import LongformerTokenizer
from datasets import EHR_Longformer_Dataset
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import torch.nn.functional as F
from models.longformernormal import LongformerPretrainNormal, LongformerFinetune
from torch.optim.lr_scheduler import LinearLR, SequentialLR, ExponentialLR, LambdaLR, CosineAnnealingWarmRestarts, ReduceLROnPlateau
from finetune_train import train
import logging
import sys
from collections import Counter
from torch.utils.data.distributed import DistributedSampler
from torch.utils.data import Sampler
from torch.optim.lr_scheduler import _LRScheduler
import math
import torch.nn.init as init
import warnings
from torch.utils.data import WeightedRandomSampler, DataLoader
import numpy as np
from collections import Counter
import random
from torch.utils.data import Dataset, DataLoader, Sampler
import torch.distributed as dist
from utils.sampler import RandomOversamplingDistributedSampler
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [7]:
import argparse
parser = argparse.ArgumentParser()

# Required parameters
parser.add_argument("--exp_name", type=str, default="mortality30")
parser.add_argument("--save_path", type=str, default="./results")
parser.add_argument("--seed", type=int, default=42)
parser.add_argument("--checkpoint_dir", type=str, default="./checkpoints")

# Model parameters
parser.add_argument("--mode", type=str, default="finetune")
parser.add_argument("--task", type=str, default="multitask")
parser.add_argument("--vocab_size", type=int, default=50265)
parser.add_argument("--itemid_size", type=int, default=3892)
parser.add_argument("--unit_size", type=int, default=81)
parser.add_argument("--gender_size", type=int, default=2)
parser.add_argument("--task_size", type=int, default=20)
parser.add_argument("--token_type_size", type=int, default=5)
parser.add_argument("--name_size", type=int, default=35)
parser.add_argument("--description_size", type=int, default=12)
parser.add_argument("--max_position_embeddings", type=int, default=10000)
parser.add_argument("--max_age", type=int, default=101)
parser.add_argument("--batch_size", type=int, default=2)
parser.add_argument("--resume", type=bool, default=False)
parser.add_argument("--pin_memory", type=bool, default=True)
parser.add_argument("--nodes", type=int, default=1)
parser.add_argument("--gpus", type=int, default=2)
parser.add_argument("--start_epoch", type=int, default=0)
parser.add_argument("--epochs", type=int, default=200)
parser.add_argument("--log_every_n_steps", type=int, default=100)
parser.add_argument("--acc", type=int, default=8)
parser.add_argument("--resume_checkpoint", type=str, default=None)
parser.add_argument("--num_workers", type=int, default=0)
parser.add_argument("--embedding_size", type=int, default=768)
parser.add_argument("--num_hidden_layers", type=int, default=1)
parser.add_argument("--num_attention_heads", type=int, default=1)
parser.add_argument("--intermediate_size", type=int, default=3072)
parser.add_argument("--learning_rate", type=float, default=1e-4)
parser.add_argument("--dropout_prob", type=float, default=0.1)
# parser.add_argument("--lora_dropout", type=float, default=0.1)
parser.add_argument("--classifier_dropout", type=float, default=0.1)
parser.add_argument("--device", type=str, default="cuda")
parser.add_argument("--gpu_mixed_precision", type=bool, default=True)
parser.add_argument("--patience", type=int, default=10)
parser.add_argument("--num_labels", type=int, default=1)
# parser.add_argument("--use_lora", action='store_true', default=False)
parser.add_argument("--gamma", type=float, default=2.0)
parser.add_argument("--beta", type=float, default=0.99)
# parser.add_argument('--lora_weight', type=int, default=2)
parser.add_argument('--classifier_weight', type=int, default=5)
parser.add_argument('--adapter_weight', type=int, default=1)
parser.add_argument("--loss", type=str, default="cross_entropy")
parser.add_argument("--pretrain", action='store_true', default=False)
parser.add_argument("--clip_interval", type=int, default=1)
parser.add_argument("--pretrain_path", type=str, default="best_pretrain_model.pth")
# parser.add_argument("--lora_layers", type=list, default=[8,9,10,11])
# parser.add_argument("--lora_r", type=int, default=4)
# parser.add_argument("--lora_alpha", type=int, default=8)
parser.add_argument("--loss_factor", type=float, default=0.5)
parser.add_argument("--mask_ratio", type=float, default=0.15)
parser.add_argument("--use_discriminator", action="store_true", help="Enable discriminator")
parser.add_argument("--regression_mode", type=bool, default=False)
parser.add_argument("--similarity_mode", type=bool, default=False)
parser.add_argument("--similarity_factor", type=float, default=0.25)
parser.add_argument("--debug", type=bool, default=False)
parser.add_argument("--freeze", action="store_true", default=False)
# parser.add_argument("--freeze", type=bool, default=False)
parser.add_argument("--oversampling_ratio", type=float, default=1.0)
parser.add_argument("--mask_mode", type=str, default="mlm")
parser.add_argument("--index", type=int, default=0)
parser.add_argument("--freeze_layers", type=int, default=0)
parser.add_argument("--num_tasks", type=int, default=7)
parser.add_argument("--num_binary_tasks", type=int, default=11)
parser.add_argument("--num_sofa_tasks", type=int, default=6)
parser.add_argument("--num_multiclass_labels", type=int, default=4)
parser.add_argument("--ablation", type=str, default=None)
parser.add_argument("--window", type=int, default=0)
parser.add_argument("--no_gap", action='store_true', default=False)
parser.add_argument("--ratio", type=int, default=100)
parser.add_argument("--num_basetask_tasks", type=int, default=1)
parser.add_argument("--num_intervention_tasks", type=int, default=1)
parser.add_argument("--value_mask_ratio", type=float, default=0.0)
parser.add_argument("--loss_mode", type=str, default="None")
parser.add_argument("--task_idx", type=lambda x: int(x) if x != "None" else None, default=None, help="Index of single binary task to run (None = run all tasks)")
parser.add_argument("--inference_mode", action='store_true', default=False)
parser.add_argument("--selected_data", type=str, default="final")
parser.add_argument("--locate", type=str, default=None)
parser.add_argument("--num_multilabel_labels", type=int, default=25, help="Use LoRA for fine-tuning")


args = parser.parse_args([])
args.attention_window = [512] * args.num_hidden_layers


In [8]:
save_path = Path(args.save_path) / args.mode
save_path.mkdir(parents=True, exist_ok=True)


tokenizer = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")

itemid2idx = pd.read_pickle("datasets/new_label2idx.pkl")
unit2idx = pd.read_pickle("datasets/new_unit2idx.pkl")
idx2label = pd.read_pickle("datasets/new_idx2label.pkl") ##############


/home/DAHS3/anaconda3/envs/sj/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
entire_dataset = pd.read_pickle("datasets/new_data_preparation/finetune_train_entire_fold0_final.pkl")
oneday_dataset = pd.read_pickle("datasets/new_data_preparation/finetune_train_24_fold0_final.pkl")

In [10]:
import numpy as np

def dataset_stats(dataset, name="dataset"):
    lengths = []
    for key, data in dataset.items():
        if isinstance(data, dict):
            # dict 내부에서 길이 있는 값만 선택
            for v in data.values():
                if hasattr(v, "__len__") and not isinstance(v, (str, bytes)):
                    lengths.append(len(v))
                    break  # 보통 하나의 주요 시계열만 있으면 break, 여러개면 제거
    lengths = np.array(lengths)
    print(f"📊 {name} 통계")
    print(f"- 샘플 수: {len(lengths)}")
    print(f"- 평균 길이: {lengths.mean():.2f}")
    print(f"- 표준편차: {lengths.std():.2f}")
    print(f"- 최소/최대: {lengths.min()} / {lengths.max()}")
    print("="*50)

dataset_stats(entire_dataset, "entire_dataset")
dataset_stats(oneday_dataset, "oneday_dataset")

📊 entire_dataset 통계
- 샘플 수: 32223
- 평균 길이: 1293.18
- 표준편차: 817.66
- 최소/최대: 84 / 4093
📊 oneday_dataset 통계
- 샘플 수: 36491
- 평균 길이: 602.51
- 표준편차: 199.86
- 최소/최대: 41 / 2837


In [21]:
import numpy as np

def dataset_stats(dataset, name="dataset"):
    lengths = []
    for key, data in dataset.items():
        if isinstance(data, dict):
            # dict 내부에서 길이 있는 값만 선택
            for v in data.values():
                if hasattr(v, "__len__") and not isinstance(v, (str, bytes)):
                    lengths.append(len(v))
                    break  # 보통 하나의 주요 시계열만 있으면 break, 여러개면 제거
    lengths = np.array(lengths)
    print(f"📊 {name} 통계")
    print(f"- 샘플 수: {len(lengths)}")
    print(f"- 평균 길이: {lengths.mean():.2f}")
    print(f"- 표준편차: {lengths.std():.2f}")
    print(f"- 최소/최대: {lengths.min()} / {lengths.max()}")
    print("="*50)

dataset_stats(entire_dataset, "entire_dataset")
dataset_stats(oneday_dataset, "oneday_dataset")

📊 entire_dataset 통계
- 샘플 수: 32223
- 평균 길이: 580.66
- 표준편차: 185.25
- 최소/최대: 6 / 2147
📊 oneday_dataset 통계
- 샘플 수: 36491
- 평균 길이: 602.51
- 표준편차: 199.86
- 최소/최대: 41 / 2837


In [6]:
args.pretrain_path = "best_pretrain_model_new_pretrain_spanmlm_0.15_gelu_14.pth"
args.pretrain = True
args.num_hidden_layers = 12
args.num_attention_heads = 6

pretrained_model = LongformerPretrainNormal(
        idx2label=idx2label, #################
        name_size=args.name_size,
        description_size=args.description_size,
        token_type_size=args.token_type_size,
        vocab_size=args.vocab_size,
        itemid_size=args.itemid_size,
        max_position_embeddings=args.max_position_embeddings,
        unit_size=args.unit_size,
        task_size=args.task_size,
        max_age=args.max_age,
        gender_size=args.gender_size,
        embedding_size=args.embedding_size,
        num_hidden_layers=args.num_hidden_layers,
        num_attention_heads=args.num_attention_heads,
        intermediate_size=args.intermediate_size,
        learning_rate=args.learning_rate,
        dropout_prob=args.dropout_prob,
        loss_factor=args.loss_factor,
        use_discriminator=args.use_discriminator,
        gpu_mixed_precision=args.gpu_mixed_precision,
    ).to(args.device)


if args.pretrain:
    pretrain_path = os.path.join("./results/", args.pretrain_path)
    checkpoint = torch.load(pretrain_path, map_location=args.device, weights_only=True)
    state_dict = checkpoint['model_state_dict']

    new_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith('module.module.'):
            new_state_dict[k[14:]] = v  
        elif k.startswith('module.'):
            new_state_dict[k[7:]] = v 
        else:
            new_state_dict[k] = v  
    # filtered_state_dict = {k: v for k, v in new_state_dict.items() if 'task_embedding' not in k}

    pretrained_model.load_state_dict(new_state_dict, strict=True)
    print("Pre-trained model loaded successfully.")

TypeError: __init__() got an unexpected keyword argument 'idx2label'

In [9]:
def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

args.freeze = True
args.freeze_layers = 8
model = LongformerFinetune(
        pretrained_model=pretrained_model,
        num_labels=args.num_labels,
        freeze_pretrained=args.freeze,
        freeze_layers=args.freeze_layers,
    ).to(args.device)


print(count_trainable_parameters(model))

36561461


In [12]:
print(pretrained_model)


LongformerPretrainNormal(
  (embeddings): EHREmbedding(
    (concept_embedding): ConceptEmbeddingwithClinicalBert(
      (embedding_model): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(28996, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=True)
                  (key): Linear(in_features=768, out_features=768, bias=True)
                  (value): Linear(in_features=768, out_features=768, bias=True)
                  (dropout): Dropout(p=0.1, inplace=False)
                

In [11]:


def count_parameters_by_layer(model):
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f"{name}: {param.numel():,}")
        
        
count_parameters_by_layer(model)

LongformerPretrainNormal(
  (embeddings): EHREmbedding(
    (concept_embedding): ConceptEmbeddingwithClinicalBert(
      (embedding_model): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(28996, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=True)
                  (key): Linear(in_features=768, out_features=768, bias=True)
                  (value): Linear(in_features=768, out_features=768, bias=True)
                  (dropout): Dropout(p=0.1, inplace=False)
                

In [7]:
print(model)

LongformerFinetune(
  (backbone): LongformerPretrainNormal(
    (embeddings): EHREmbedding(
      (concept_embedding): ConceptEmbeddingwithClinicalBert(
        (embedding_model): BertModel(
          (embeddings): BertEmbeddings(
            (word_embeddings): Embedding(28996, 768, padding_idx=0)
            (position_embeddings): Embedding(512, 768)
            (token_type_embeddings): Embedding(2, 768)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (encoder): BertEncoder(
            (layer): ModuleList(
              (0-11): 12 x BertLayer(
                (attention): BertAttention(
                  (self): BertSdpaSelfAttention(
                    (query): Linear(in_features=768, out_features=768, bias=True)
                    (key): Linear(in_features=768, out_features=768, bias=True)
                    (value): Linear(in_features=768, out_features=768, bias=True)
     